# ARC-AGI-2 — Test-Time Bayesian Program Induction (offline submission)

This notebook produces a `submission.json` for the ARC Prize 2026 ARC-AGI-2
competition using the **offline** solver from the `arc_solver` package:
fitted analytic proposals + MDL-ordered enumerative program search, with a
pass@2-aware posterior-mode selector. It runs **fully offline** (no internet,
no API calls) inside the Kaggle sandbox.

**Method in one line:** for each task we induce a short program (over a compact
grid DSL) that reproduces *all* demonstration pairs, weight admissible programs
by a Minimum-Description-Length prior, and emit the posterior-mode output as
`attempt_1` and the most probable *disagreeing* output as `attempt_2`.

**Kaggle input (exact path this notebook expects at rerun):**
`/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json` — a single
combined JSON file `{"<task_id>": {"train": [...], "test": [...]}}` covering
all 240 unseen test tasks (test pairs have no `"output"`; it is hidden). This
notebook does **not** glob a directory of per-task files — the competition
does not mount the data that way.

**Internet must be turned OFF** for this notebook (Settings → Internet → Off).
The competition sandbox has no network access at scoring time regardless, but
turning it off locally too is what lets you verify the notebook will actually
run under the same constraints it will be rerun under.

To run on Kaggle, attach this repository's `src/` directory as a dataset
(e.g. `arc-solver-src`) so `arc_solver` is importable — see `DEPLOYMENT.md` for
the exact steps (Kaggle Dataset attach, or inlining the package). The cell
below searches common locations automatically.


In [ ]:
import os, sys, glob

# Locate the arc_solver package no matter how the Kaggle dataset is mounted.
# Kaggle has used several conventions: /kaggle/input/arc-solver-src,
# /kaggle/input/datasets/<owner>/arc-solver-src, with the package either at the
# dataset root (arc_solver/...) or under a src/ wrapper. Rather than hardcode
# every path, we try the common ones fast, then glob /kaggle/input for the
# package as a robust fallback.
def _find_arc_solver():
    fixed = [
        '/kaggle/input/arc-solver-src/src',
        '/kaggle/input/arc-solver-src',
        '/kaggle/input/arc-solver/src',
        '/kaggle/input/arc-solver',
        os.path.join(os.getcwd(), 'src'),
        os.getcwd(),
        os.path.join(os.path.dirname(os.getcwd()), 'src'),
    ]
    for c in fixed:
        if os.path.isdir(os.path.join(c, 'arc_solver')):
            return c
    # Robust fallback: find arc_solver/__init__.py wherever it was mounted.
    for base in ('/kaggle/input', os.getcwd()):
        hits = glob.glob(os.path.join(base, '**', 'arc_solver', '__init__.py'), recursive=True)
        if hits:
            return os.path.dirname(os.path.dirname(min(hits, key=len)))
    return None

_root = _find_arc_solver()
if _root is None:
    raise RuntimeError(
        'arc_solver not found under /kaggle/input or CWD; attach the '
        'arc-solver-src dataset (see DEPLOYMENT.md).'
    )
sys.path.insert(0, _root)
print('arc_solver from', _root)

from arc_solver.cli.run_eval import build_engine
from arc_solver.io.loader import load_challenges
from arc_solver.io.submission import SubmissionWriter

In [ ]:
# Locate the combined-challenges JSON. Kaggle file naming has varied
# (arc-agi_test_challenges.json / arc-agi_evaluation_challenges.json, with "_"
# or "-"), and the hidden test file only appears at rerun. So we glob
# /kaggle/input for any *challenges*.json (excluding *solutions*), preferring
# test > evaluation > training; then fall back to a local dry-run file.
def _find_challenges():
    def rank(p):
        b = os.path.basename(p).lower()
        for i, key in enumerate(('test', 'evaluation', 'training')):
            if key in b:
                return (i, len(p))
        return (3, len(p))
    cands = [
        p for p in glob.glob('/kaggle/input/**/*challenges*.json', recursive=True)
        if 'solution' not in os.path.basename(p).lower()
    ]
    if cands:
        return min(cands, key=rank)
    for p in ('data/kaggle_dry_run_challenges.json', '../data/kaggle_dry_run_challenges.json'):
        if os.path.isfile(p):
            return p
    return None

CHALLENGES_PATH = _find_challenges()
if CHALLENGES_PATH is None:
    tree = []
    for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
        depth = root.count(os.sep) - '/kaggle/input'.count(os.sep)
        if depth <= 2:
            tree.append(root + os.sep)
        for f in files:
            if f.lower().endswith('.json'):
                tree.append(os.path.join(root, f))
        if len(tree) > 80:
            break
    raise FileNotFoundError(
        'No *challenges*.json found under /kaggle/input. The competition data is '
        'not attached. In the notebook editor: Add Data -> Competitions -> search '
        '"ARC Prize 2026" -> add the ARC-AGI-2 competition (or open this notebook '
        'from the competition page, which auto-attaches it). Tree under '
        '/kaggle/input:\n  ' + '\n  '.join(tree[:80] or ['(empty)'])
    )

tasks = load_challenges(CHALLENGES_PATH)
print('challenges file:', CHALLENGES_PATH, '| tasks:', len(tasks))

In [ ]:
import time

# Offline engine: fitted proposals first, then bounded enumerative search.
# Fully offline — numpy only, no network, no LLM calls.
engine = build_engine(max_seconds=10.0, max_programs=20000, max_depth=4)
writer = SubmissionWriter()

TOTAL_BUDGET_S = 11 * 3600   # wall-clock guard, well under the 12h Kaggle limit
start = time.monotonic()
solved = 0

for i, task in enumerate(tasks):
    if time.monotonic() - start > TOTAL_BUDGET_S:
        # Out of time: emit a safe identity/mirror fallback so no task is missing.
        attempts = []
        for p in task.test:
            g = p.input
            mirror = tuple(tuple(reversed(r)) for r in g)
            attempts.append((g, mirror if mirror != g else g))
        writer.add(task.task_id, attempts)
        continue
    result = engine.solve(task)
    solved += int(result.solved_train)
    writer.add(task.task_id, list(result.attempts))
    if (i + 1) % 20 == 0:
        print(f'{i + 1}/{len(tasks)} tasks, {solved} with a program, '
              f'{time.monotonic() - start:.0f}s')

# Write to the exact path Kaggle scores; fall back to a local path off-Kaggle.
OUT_PATH = '/kaggle/working/submission.json' if os.path.isdir('/kaggle/working') else 'submission.json'
writer.write(OUT_PATH)
assert len(writer) == len(tasks), 'every task must be present in the submission'
print('wrote', OUT_PATH, ':', len(writer), 'tasks,', solved, 'with an admissible program')


The full method, theory, and ablations are described in the accompanying
write-up (`WRITEUP.md`). The offline path above is the reproducible core; an
optional LLM-amortised proposer (`arc_solver.search.llm_proposer`) raises
accuracy when an API key is available, but **cannot run inside the Kaggle
no-internet sandbox** — it is disabled here. See `SECURITY.md` and
`docs/RUNBOOK.md` §4 for the gated rollout of that path outside Kaggle.
